# GEMS P6 — fast track to the leaderboard

Goal: the strongest **valid** submission before 2026-12-03 under 12 T4 GPU-h / 20 CPU-h. The B0+B1 fallback
`m9_fallback_fusion_thin.tif` stays the champion until a candidate passes its gate. Guide: `docs/p6/P6_FAST_TRACK.md`.

| section | workstream | device | exit gate |
|---|---|---|---|
| 1 | W0 leaderboard probes A-D | CPU | which evidence wins: broad discovery / near-known / budget |
| 2 | W1 seeds + 180° TTA | GPU (train, TTA) + CPU | disc ≥ fallback + 0.005, geo and worst zone no regression |
| 3 | W2 structural stacker | CPU | disc ≥ 0.167, worst ≥ 0.126, geo ≥ ref − 0.005 |
| 4 | W3 M9 throughput | GPU | ≤ 90 s / epoch, else stop M9 |
| 5 | W4 DEM pilot | CPU | ≥ 30 % trace recall @ 40 km/100 km², ≥ 3 new systems, alignment |
| 6 | W5 near-known completion | CPU | only if W0 says `near_known_completion` |
| 7 | W6 final selection + GPKG | CPU | ordered criteria |

**Rules.** Nothing is uploaded by code. Every section reads only persisted outputs, so after a disconnect run section 0,
then any section. Each workstream writes its wall time to `manifests/compute_ledger.csv` and refuses to start when its
budget is spent. Existing M9 files and checkpoints are never overwritten.

## 0. Configuration, Drive and context

Same workflow as the M9 / G02 notebooks. Prerequisites on Drive: M9 section 01 (`baseline/baseline_selection.json`,
`submissions/m9_fallback_fusion_thin.tif`) and, for W2's structural features, a G02 run.
Use a **GPU runtime** only for W1 (training / TTA) and W3; everything else runs on CPU.

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'p6-fast-track',                          # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,                                 # needed for W1 training / TTA and W3 only
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}   # W4: download only after approval
PRIVATE_REPO = False

M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}   # opens the M9 context; no M9 section is run here
P6_CONFIG = {'overwrite_existing': False}                        # overrides of configs/p6/fast_track.yaml

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import display
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
print('P6 outputs ->', p6.ws.root)

def tab(sub, name, n=40):
    t = p6.ws.load_table(name, sub)
    display(t.head(n)) if len(t) else print('empty / not produced:', f'{sub}/{name}')
    return t

display(p6.budget.table())

## 1. W0 — leaderboard probes (no GPU)

* **A** the existing fallback file (read-only), **B** A's rule and pixel count + 300 m suppression ring,
  **C** A with the `disc_official` nested area fraction (skipped if equal to A's), **D** diagnostic: every
  template-finite non-known pixel = 1.
* Ring = *suppression*. B > A: near-known predictions are net false positives (broad discovery). B < A: public new
  truth lies next to known traces (W5). One delta is evidence about the public subset only.
* D brackets the public new-fault density (see `gems_bench/p6/probes.py`); it is never a candidate.
* **Upload manually** (3 scored submissions per rolling 7 days): A, B, C this week; D only if a slot is left.

In [ ]:
log = p6.w0_probes()
display(log)
tab('probes', 'probe_plan')
print('files to upload:', p6.ws.root / 'submissions', '(A stays in the M9 submissions folder)')

### 1b. Record public scores, then interpret

Type the public score of each uploaded file below (matched by the first characters of its SHA256), run the cell, then
read the interpretation. Missing scores stay PENDING; nothing is imputed. Copy the updated
`outputs/p6_fast_track/probes/leaderboard_log.csv` into the repository's `reports/p6/leaderboard_log.csv` when you commit.

In [ ]:
SCORES = {}            # {'<sha256 prefix>': 0.1234, ...}  public leaderboard scores, typed by hand

from gems_bench.p6 import probes
L = probes.read_log(p6)
for pre, v in SCORES.items():
    hit = L.sha256.astype(str).str.startswith(pre)
    assert hit.sum() == 1, f'{pre}: {hit.sum()} matches'
    L.loc[hit, 'public_score'] = float(v)
probes.update_logs(p6, L)
display(probes.read_log(p6))
res = p6.w0_interpret()
print('VERDICT:', res['verdict'])
for f in res['findings']:
    print(' -', f)
print('missing:', res['missing_scores'], '|', res['caveat'])

## 2. W1 — seeds + 180° TTA (≤ 4 GPU-h)

`seeds.plan` lists existing members and the extra seeds that fit the remaining W1 GPU budget (measured run cost when
available). Set `TRAIN_SEEDS = True` to train them (resumable experiments). TTA maps are re-predicted by the same
held-out checkpoint (`cv_fold<k>` / `disc<d>`), so they stay honest. With one member per family and no TTA, the
`seeds` variant must reproduce the fallback's numbers: check that first.

In [ ]:
from gems_bench.p6 import seeds
display(seeds.plan(p6))
TRAIN_SEEDS = False
w1 = p6.w1_seeds(train=TRAIN_SEEDS)
display(w1['summary'])
print(json.dumps(w1['candidate'], indent=1, default=str)[:3000])

## 3. W2 — structural stacker (CPU, ≤ 6 h)

Honest B0 / B1 ranks + G02-C / G02-A (+ G01 persistence when present) + potential-field gradients / tilt / strike
agreement + geometry of the *permitted* known map; LightGBM fitted leave-one-zone-out with a 2 km buffer; nested
(blend, rep, frac); unique systems vs the fallback. Provenance of every channel is asserted (`LeakageError`).
The gate decides once, with no re-tuning. The full-region file is written only on PROMOTE (it needs full-grid G02
maps; their tiles are resumable).

In [ ]:
w2 = p6.w2_stack(full_region=True)
for n in ('provenance', 'crossfit_audit', 'per_zone', 'unique_systems', 'feature_importance'):
    tab('stack', n)
print(json.dumps(w2['candidate'], indent=1, default=str)[:3000])

## 4. W3 — M9 throughput (GPU, ≤ 2 h)

Component timings on the real M9 objects (patch read, scaling, context, corruption, batch, host to device, forward /
backward, inner validation) for memory-mapped vs RAM cube and pageable vs pinned batches. Decision:
`PERMIT_ONE_CORE` (≤ 90 s / epoch: one `core` model on limited folds) or `STOP_M9`.\nThe next cell stages **only existing** M9 registry artifacts from Drive if absent locally, then checks channel compatibility. No M9 section 02 rerun.\n

In [ ]:
# W3 prerequisite: restore existing M9 feature selection from Drive (read-only).
# M9 .load() doesn't necessarily stage the section-02 registry; W3 reads m9.ws.
from gems_bench.m9.features import selected_registry, channel_groups

if selected_registry(p6.m9).empty:
    p6.stage_m9([
        "features/selected_registry.parquet",
        "features/selected_registry.csv",
        "features/inventory.parquet",
        "features/inventory.csv",
        "spectral/gate1.parquet",
        "spectral/gate1.csv",
        "spectral/spectral_registry.parquet",
        "spectral/spectral_registry.csv",
    ])

reg = selected_registry(p6.m9)
if reg.empty or "selected" not in reg.columns or not reg["selected"].astype(bool).any():
    raise RuntimeError(
        "W3 blocked: selected M9 feature registry is absent or invalid after Drive staging. "
        "Restore matching M9 section-02 artifacts; do not rerun section 02 blindly."
    )
variant_cfg = p6.m9.variant_cfg(p6.cfg["throughput"]["variant"])
groups = channel_groups(p6.m9, variant_cfg)
if not any(groups.values()):
    raise RuntimeError("W3 blocked: no channels resolved from the restored M9 registries.")
print(f"W3 dependencies ready: {len(reg)} registry rows, "
      f"{int(reg['selected'].astype(bool).sum())} selected, "
      f"channel groups = { {k: len(v) for k, v in groups.items()} }")


In [ ]:
w3 = p6.w3_throughput()
display(w3)
print(json.loads((p6.ws.root / 'throughput' / 'decision.json').read_text()))

## 5. W4 — DEM pilot (CPU, ≤ 6 h)

Reuses the cached M7 DEM mosaic; only if it is absent, runs M7 sections 05 / 06 with the pilot capped at 30 tiles (set
`DEM_CONFIG['download']` only after reading M7's storage estimate). Coverage masks the gate and is never a feature.

In [ ]:
w4 = p6.w4_dem()
print(json.dumps(w4, indent=1, default=str))
tab('dem', 'systems')

## 6. W5 — near-known completion (conditional)

Runs only when the W0 verdict is `near_known_completion`; otherwise it returns BLOCKED and does nothing.

In [ ]:
w5 = p6.w5_completion()
print(json.dumps(w5, indent=1, default=str)[:3000])

## 7. W6 — final selection and lineaments

Fallback + every candidate that passed its gate; order: discovery DTI, worst zone, geographic DTI, unique systems,
public score (tie-breaker). Writes `selection/decision.json` and `vectors/final_lineaments.gpkg` (segment confidence
is a heuristic rank, not a probability). Freeze by 2026-11-25; final upload by 2026-11-30.

In [ ]:
w6 = p6.w6_select()
print(json.dumps(w6, indent=1, default=str))
tab('selection', 'candidates')
tab('selection', 'elimination_trace')
display(p6.budget.table())